# ER triage acuity classification model
CSCI5622 Project \
Name: Sabrina Touch

### 1. Install packages, load data

In [ ]:
#%pip install pandas
#%pip install numpy
#%pip install scikit-learn

In [43]:
import numpy as np
import pandas as pd

from sklearn.metrics import recall_score, confusion_matrix
from sklearn.model_selection import train_test_split

# Load in data, set SEED
SEED = 42
data = pd.read_csv("./nhamcs_data_2018_22.csv")

print("Imported libraries, read in csv")

Imported libraries, read in csv


In [60]:
target_col = 'target_triage_acuity'

# Temporal holdout: Isolate 2022 as the untouched test set, and use 2018-2021 as the training set
mask_2022 = data['year'] == 2022
training_data_2018_2021 = data[~mask_2022] # 2018-2021 data for training
testing_data_2022 = data[mask_2022] # 2022 data for testing

### 2. Set baseline

In [ ]:
# Determine baseline score where it is the Recall achieved by applying the rule "If EMS arrival = yes, predict critical"

# Convert to binary: 1 if critical, 0 if not critical
# Note: we are using the binary classification of critical vs. non-critical for the baseline model, where critical is defined as triage acuity levels 1 or 2.
y_test_binary = testing_data_2022[target_col].isin([1, 2]).astype(int) 

baseline_predictions = (testing_data_2022['ems_arrival'] == 'Yes').astype(int)
baseline_recall = recall_score(y_test_binary, baseline_predictions)

tn, fp, fn, tp = confusion_matrix(y_test_binary, baseline_predictions).ravel()
baseline_fpr = fp / (fp + tn)

In [62]:
print(f"EMS Arrival Baseline Recall Score: {baseline_recall:.4f} (at FPR: {baseline_fpr:.4f})")

EMS Arrival Baseline Recall Score: 0.3541 (at FPR: 0.1534)


### 3. Split and stratify

In [71]:
# Separating features (X) and target (y) for training sets
X_train_full = training_data_2018_2021.drop(target_col, axis=1)
y_train_full = training_data_2018_2021[target_col]

# Separating features (X) and target (y) for test sets
X_test = testing_data_2022.drop(target_col, axis=1)
y_test = testing_data_2022[target_col]

# Stratify the split to maintain class distribution in train and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, 
    y_train_full, 
    test_size=0.2, 
    random_state=SEED, 
    stratify=y_train_full
)

# Convert target variable to binary for training and validation sets
# Note: Binary classifying after the stratifying split to ensure that the class distribution of all 1-5 acuity levels is maintained.
y_train_binary = y_train.isin([1, 2]).astype(int)
y_val_binary = y_val.isin([1, 2]).astype(int)

In [ ]:
assert X_train.shape[0] + X_val.shape[0] == X_train_full.shape[0], "Train and validation split sizes do not add up to full training set size."
assert y_train.shape[0] + y_val.shape[0] == y_train_full.shape[0], "Train and validation split sizes do not add up to full training set size."
print(f"Training set size: {X_train.shape[0]} samples")

Training set size: 38333 samples


### 4. Logistic Regression

### AI Appendix
> 1. 